#### Imports

In [1]:
import json
import random
import numpy as np
import yaml 
import torch

from torch import nn
from torch.utils.data import DataLoader
from pathlib import Path
from datetime import datetime
from uuid import uuid4

In [2]:
from src.data.datasets.sen1floods11 import Sen1Floods11Dataset
from src.data.statistics import get_or_compute_stats
from src.data.preprocessing import NormalizeFromStats
from src.data.transforms import random_crop
from src.encoders.croma.croma import CROMAEncoder
from src.adapters.token_to_spatial import TokenToSpatial
from src.heads.fcn import FCNHead
from src.models.dense_prediction import DensePredictionModel
from src.evaluation.evaluator import evaluate, save_evaluation_report
from training.trainer import train_one_epoch
from training.checkpoints import save_checkpoint, load_checkpoint

#### Reading YAML configuration file

In [3]:
config_path = Path("configs/croma/croma_s1s2_fcn_all_hand_labels.yaml")
with config_path.open(encoding="utf-8") as file:
    config = yaml.safe_load(file)

In [4]:
data_cfg = config["dataset"]
model_cfg = config["model"]
train_cfg = config["training"]
eval_cfg = config["evaluation"]
preprocess_cfg = config["preprocessing"]
seed = config["experiment"]["seed"]

loss_cfg = train_cfg["loss"]
optimizer_cfg = train_cfg["optimizer"]

print(data_cfg)

{'name': 'sen1floods11', 'modalities': ['sar', 'optical'], 'train_csv': 'data/sen1floods11/splits/all_hand_labels/flood_train_data.csv', 'val_csv': 'data/sen1floods11/splits/all_hand_labels/flood_valid_data.csv', 'test_csv': 'data/sen1floods11/splits/all_hand_labels/flood_test_data.csv', 'bolivia_csv': 'data/sen1floods11/splits/all_hand_labels/flood_bolivia_data.csv'}


In [5]:
seed = config["experiment"]["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
modalities = tuple(data_cfg["modalities"])
croma_modality = model_cfg["modality"]

expected = {"SAR": {"sar"}, "optical": {"optical"}, "both": {"sar", "optical"}}

if set(modalities) != expected[croma_modality]:
    raise ValueError("Dataset and encoder modalities disagree")

pretrained_path = Path(model_cfg["checkpoint"])

crop_size = preprocess_cfg["crop_size"]
batch_size = train_cfg["batch_size"]
epochs = train_cfg["epochs"]
validate_every = train_cfg.get("validate_every", 5)
learning_rate = optimizer_cfg["learning_rate"]
weight_decay = optimizer_cfg["weight_decay"]
freeze_encoder = model_cfg["freeze_encoder"]
class_weights = loss_cfg.get("class_weights")
ignore_index = loss_cfg["ignore_index"]

if model_cfg["image_resolution"] != crop_size or eval_cfg["tile_size"] != crop_size:
    raise ValueError("Keep CROMA image_resolution, crop_size and tile_size equal here")

In [7]:
split_paths = {
    split: Path(data_cfg[f"{split}_csv"])
    for split in ("train", "val", "test", "bolivia")
}
assert pretrained_path.is_file(), pretrained_path
assert all(path.is_file() for path in split_paths.values())
print("Device:", device)

Device: cuda


#### Loading the datasets and checking the splits

In [8]:
raw_datasets = {
    split: Sen1Floods11Dataset(
        split_csv=path,
        modalities=modalities,
        preprocess_func=None,
    )
    for split, path in split_paths.items()
}

seen = {}
for split, dataset in raw_datasets.items():
    print(split, len(dataset))
    for paths in dataset.data_list:
        chip = paths["target"].name
        if chip in seen:
            raise ValueError(f"Chip appears in {seen[chip]} and {split}: {chip}")
        seen[chip] = split

sample = raw_datasets["train"][0]
print(sample["chip_id"])
print(sample["sar"].shape, sample["optical"].shape, sample["target"].shape)

train 252
val 89
test 90
bolivia 15
Ghana_103272
torch.Size([2, 512, 512]) torch.Size([13, 512, 512]) torch.Size([512, 512])


#### Compute normalisation statistics from training images only

In [9]:
def raw_training_dataset(selected_modalities):
    return Sen1Floods11Dataset(
        split_csv=split_paths["train"],
        modalities=selected_modalities,
        preprocess_func=None,
    )

statistics = None
if preprocess_cfg["normalization"]["statistics"] == "training_dataset":
    statistics = get_or_compute_stats(
        dataset_factory=raw_training_dataset,
        modalities=modalities,
        split_csv=split_paths["train"],
        dataset_name=data_cfg["name"],
        cache_dir="data/statistics",
    )
normalizer = NormalizeFromStats.from_config(config, training_statistics=statistics)


Loading cached statistics for sar: data\statistics\sen1floods11\57be4dc440bf\sar_0321578226838c8d.json
Loading cached statistics for optical: data\statistics\sen1floods11\57be4dc440bf\optical_b2316b09fcd43301.json


#### Define training preprocesing and create DataLoaders

In [10]:
class TrainingPreprocess:
    def __init__(self, normalizer, crop_size):
        self.normalizer = normalizer
        self.crop_size = crop_size

    def __call__(self, sample):
        sample = random_crop(sample, size=self.crop_size)
        return self.normalizer(sample)

train_dataset = Sen1Floods11Dataset(
    split_csv=split_paths["train"],
    modalities=modalities,
    preprocess_func=TrainingPreprocess(normalizer, crop_size),
)
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=train_cfg.get("num_workers", 0),
)

evaluation_loaders = {
    split: DataLoader(
        Sen1Floods11Dataset(
            split_csv=split_paths[split],
            modalities=modalities,
            preprocess_func=normalizer,
        ),
        batch_size=eval_cfg["batch_size"], shuffle=False,
        num_workers=train_cfg.get("num_workers", 0),
    )
    for split in ("val", "test", "bolivia")
}

#### Create the encoder

In [11]:
encoder = CROMAEncoder(
    checkpoint_path=pretrained_path,
    size=model_cfg["size"],
    modality=croma_modality,
    image_resolution=model_cfg["image_resolution"],
    freeze=freeze_encoder,
)

Initializing SAR encoder
Initializing optical encoder
Initializing joint SAR-optical encoder


#### Create the adapter and head, then assemble the model

In [12]:
adapter = TokenToSpatial(
    feature_key=config["adapter"]["feature_key"],
    grid_size=config["adapter"].get("grid_size"),
)
head = FCNHead(
    in_channels=config["head"]["in_channels"],
    num_classes=config["head"]["num_classes"],
)
model = DensePredictionModel(encoder=encoder, adapter=adapter, head=head)
model = model.to(device)

print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Trainable parameters: 2360322


#### Create the loss, optimiser and optional scheduler

In [13]:
if loss_cfg["name"] != "cross_entropy":
    raise ValueError("Add an explicit criterion implementation for this loss")

weights = None if class_weights is None else torch.tensor(
    class_weights, dtype=torch.float32, device=device
)

criterion = nn.CrossEntropyLoss(
    weight=weights, ignore_index=ignore_index, reduction="mean",
)

optimizer_types = {
    "adamw": torch.optim.AdamW,
    "adam": torch.optim.Adam,
    "sgd": torch.optim.SGD,
}

optimizer_name = optimizer_cfg["name"]
optimizer_options = {"lr": learning_rate, "weight_decay": weight_decay}

if optimizer_name == "sgd":
    optimizer_options["momentum"] = optimizer_cfg.get("momentum", 0.9)
optimizer = optimizer_types[optimizer_name](
    [p for p in model.parameters() if p.requires_grad], **optimizer_options
)

scheduler_cfg = train_cfg.get("scheduler") or {"name": "none"}

if scheduler_cfg["name"] == "none":
    scheduler = None
elif scheduler_cfg["name"] == "cosine_annealing_warm_restarts":
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=int(scheduler_cfg["t0_epochs"] * len(train_loader)),
        T_mult=scheduler_cfg.get("t_mult", 1),
        eta_min=scheduler_cfg.get("eta_min", 0.0),
    )
else:
    raise ValueError("Add the scheduler's construction and stepping policy explicitly")


#### Prepare the run folder and evaluation call

In [14]:
run_id = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
run_dir = Path(config["output"]["directory"]) / run_id
run_dir.mkdir(parents=True, exist_ok=False)

settings = {
    "seed": seed,
    "modalities": list(modalities),
    "croma_modality": croma_modality,
    "pretrained_path": str(pretrained_path),
    "encoder_size": model_cfg["size"],
    "freeze_encoder": freeze_encoder,
    "crop_size": crop_size,
    "batch_size": batch_size,
    "epochs": epochs,
    "validate_every": validate_every,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "class_weights": class_weights,
    "ignore_index": ignore_index,
    "scheduler": scheduler_cfg,
    "evaluation_tile_size": eval_cfg["tile_size"],
    "evaluation_stride": eval_cfg.get("stride"),
    "split_paths": {key: str(value) for key, value in split_paths.items()},
}
(run_dir / "config.yaml").write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
(run_dir / "settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
normalization_settings = {
    "statistics": normalizer.statistics,
    "method": normalizer.method,
    "band_indices": normalizer.band_indices,
    "use_8_bit": normalizer.use_8_bit,
}
(run_dir / "normalization.json").write_text(
    json.dumps(normalization_settings, indent=2), encoding="utf-8"
)

# Record the identity of the pretrained weights and split files.
import hashlib

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

(run_dir / "provenance.json").write_text(json.dumps({
    "pretrained_sha256": file_sha256(pretrained_path),
    "split_sha256": {key: file_sha256(path) for key, path in split_paths.items()},
}, indent=2), encoding="utf-8")

def evaluate_split(loader):
    return evaluate(
        model=model,
        dataloader=loader,
        device=device,
        num_classes=2,
        criterion=criterion,
        ignore_index=ignore_index,
        tile_size=eval_cfg["tile_size"],
        stride=eval_cfg.get("stride"),
        tile_batch_size=eval_cfg["tile_batch_size"],
        input_keys=modalities,
        return_per_sample=eval_cfg.get("return_per_sample", True),
        sample_id_key=eval_cfg.get("sample_id_key", "chip_id"),
        class_names=eval_cfg.get("class_names", ["non_water", "water"]),
    )

#### Train, validate and save the best checkpoints

In [15]:
if epochs < 1 or validate_every < 1:
    raise ValueError("epochs and validate_every must be positive")

print(f"validating every: {validate_every} epochs")

history = []
best_score = float("-inf")
best_path = run_dir / "checkpoints/best.pt"
last_path = run_dir / "checkpoints/last.pt"

# Frozen backbone: save only trainable components to avoid duplicating weights.
# Unfrozen backbone: save the full model.
components = ["adapter", "head"] if freeze_encoder else None

for epoch in range(1, epochs + 1):
    lr_start = optimizer.param_groups[0]["lr"]
    train_loss = train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        scheduler=scheduler,
        input_keys=modalities,
    )
    row = {"epoch": epoch, "train_loss": train_loss, "lr_start": lr_start}
    validation_summary = None

    if epoch % validate_every == 0 or epoch == epochs:
        validation = evaluate_split(evaluation_loaders["val"])
        save_evaluation_report(
            run_dir / "metrics" / f"validation_{epoch:03d}.json",
            validation,
            metadata={"split": "val", "epoch": epoch},
        )
        validation_summary = {key: value for key, value in validation.items()
                              if key != "per_sample"}
        row["validation"] = validation_summary
        metric_name = eval_cfg.get("checkpoint_metric", "miou")
        if metric_name == "iou":
            score = validation["iou_per_class"][eval_cfg.get("checkpoint_class", 1)]
        elif metric_name == "miou":
            score = validation["miou"]
        else:
            raise ValueError("This example selects by iou or miou (maximize)")

        if score is not None and score > best_score:
            best_score = score
            save_checkpoint(
                best_path, model, epoch,
                optimizer=optimizer, scheduler=scheduler,
                metrics=validation_summary, config=settings, components=components,
            )

    save_checkpoint(
        last_path, model, epoch,
        optimizer=optimizer, scheduler=scheduler,
        metrics=validation_summary, config=settings, components=components,
    )
    history.append(row)
    (run_dir / "metrics").mkdir(exist_ok=True)
    (run_dir / "metrics/history.json").write_text(
        json.dumps(history, indent=2, allow_nan=False), encoding="utf-8"
    )
    message = f"Epoch {epoch:03d}/{epochs} | LR {lr_start:.2e} | Train loss {train_loss:.4f}"
    if validation_summary is not None:
        message += (f" | Water IoU {validation_summary['iou_per_class'][1]}"
                    f" | mIoU {validation_summary['miou']}")
    print(message, flush=True)

validating every: 10 ephocs
Epoch 001/100 | LR 5.00e-04 | Train loss 0.2916
Epoch 002/100 | LR 4.88e-04 | Train loss 0.2033
Epoch 003/100 | LR 4.52e-04 | Train loss 0.1748
Epoch 004/100 | LR 3.97e-04 | Train loss 0.1630
Epoch 005/100 | LR 3.27e-04 | Train loss 0.1668
Epoch 006/100 | LR 2.50e-04 | Train loss 0.1387
Epoch 007/100 | LR 1.73e-04 | Train loss 0.1457
Epoch 008/100 | LR 1.03e-04 | Train loss 0.1546
Epoch 009/100 | LR 4.77e-05 | Train loss 0.1569
Epoch 010/100 | LR 1.22e-05 | Train loss 0.1584 | Water IoU 0.6764909251066716 | mIoU 0.810122822957817
Epoch 011/100 | LR 5.00e-04 | Train loss 0.1585
Epoch 012/100 | LR 4.97e-04 | Train loss 0.1466
Epoch 013/100 | LR 4.88e-04 | Train loss 0.1491
Epoch 014/100 | LR 4.73e-04 | Train loss 0.1535
Epoch 015/100 | LR 4.52e-04 | Train loss 0.1846
Epoch 016/100 | LR 4.27e-04 | Train loss 0.1658
Epoch 017/100 | LR 3.97e-04 | Train loss 0.1514
Epoch 018/100 | LR 3.63e-04 | Train loss 0.1323
Epoch 019/100 | LR 3.27e-04 | Train loss 0.1559
Epoc

#### Reload the best weights and evaluate held-out chips

In [16]:
if not best_path.is_file():
    raise RuntimeError("No valid best checkpoint was selected")

checkpoint_info = load_checkpoint(best_path, model, map_location=device)
checkpoint_hash = file_sha256(best_path)
print("Selected epoch:", checkpoint_info["epoch"])

# Unique folder avoids overwriting earlier evaluations if you rerun this cell.
evaluation_dir = run_dir / "metrics" / ("evaluation_" + uuid4().hex[:8])
for split in ("test", "bolivia"):
    results = evaluate_split(evaluation_loaders[split])
    save_evaluation_report(
        evaluation_dir / f"{split}.json",
        results,
        metadata={
            "split": split,
            "checkpoint": str(best_path),
            "checkpoint_sha256": checkpoint_hash,
            "selected_epoch": checkpoint_info["epoch"],
            "seed": seed,
        },
    )
    print(split, "mIoU:", results["miou"], "water IoU:", results["iou_per_class"][1])

Selected epoch: 20
test mIoU: 0.837908759417278 water IoU: 0.7243173682994022
bolivia mIoU: 0.7735809608565666 water IoU: 0.6269393642398802


#### Read the results

In [17]:
report = json.loads((evaluation_dir / "test.json").read_text())
print("mIoU (%):", 100 * report["miou"])
print("Water IoU (%):", 100 * report["iou_per_class"][1])
print("Water precision:", report["precision_per_class"][1])
print("Water recall:", report["recall_per_class"][1])
print("Water F1:", report["f1_per_class"][1])
print("Pixel accuracy:", report["pixel_accuracy"])
print("Confusion matrix:", report["confusion_matrix"])

recall = report["recall_per_class"][1]
precision = report["precision_per_class"][1]
omission = None if recall is None else 1 - recall
commission = None if precision is None else 1 - precision
chip_values = [sample["iou_per_class"][1] for sample in report.get("per_sample", [])
               if sample["iou_per_class"][1] is not None]
mean_per_chip_water_iou = sum(chip_values) / len(chip_values) if chip_values else None
print("Omission:", omission, "Commission:", commission)
print("Mean per-chip water IoU:", mean_per_chip_water_iou)
print("Chips with defined water IoU:", len(chip_values))

mIoU (%): 83.7908759417278
Water IoU (%): 72.43173682994022
Water precision: 0.7849070500355567
Water recall: 0.9036900729940092
Water F1: 0.8401207128288175
Pixel accuracy: 0.9569818096054917
Confusion matrix: [[17315787, 635479], [247141, 2318960]]
Omission: 0.09630992700599084 Commission: 0.21509294996444328
Mean per-chip water IoU: 0.46701415502344523
Chips with defined water IoU: 86
